# My Quick Reference: Multi-Head Attention + Data Loading

**Date**: February 21, 2026

I'm putting together a compact, runnable notebook that combines:
- The data loading pipeline from Chapter 2 (so I have real batched input)
- Both MHA implementation strategies from Chapter 3

This is my "cheat sheet" — everything I need to get input embeddings into an MHA module in one place.

**Attribution**: Concepts from *Build a Large Language Model From Scratch* (Raschka). Code is my own reconstruction.

In [ ]:
from importlib.metadata import version
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## Part 1: Data Loading (Ch02 recap)

I need a real batch of token embeddings as input to the attention module.
The sliding-window dataset from ch02 gives me that.

In [ ]:
class GPTDatasetV1(Dataset):
    """Sliding-window dataset: produces (input_ids, target_ids) pairs."""

    def __init__(self, txt: str, tokenizer, max_length: int, stride: int) -> None:
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.input_ids: list[torch.Tensor] = []
        self.target_ids: list[torch.Tensor] = []

        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt: str, batch_size: int = 4, max_length: int = 256,
                      stride: int = 128, shuffle: bool = True) -> DataLoader:
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


with open("small-text-sample.txt", "r", encoding="utf-8") as fh:
    raw_text = fh.read()

print(f"Loaded {len(raw_text)} characters")

In [ ]:
# Embedding layers
VOCAB_SIZE     = 50_257   # GPT-2 vocabulary size
EMBED_DIM      = 256      # output embedding dimension
CONTEXT_LENGTH = 1_024    # maximum supported sequence length
MAX_LENGTH     = 4        # window size for this demo (small for speed)

token_embed_layer = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
pos_embed_layer   = nn.Embedding(CONTEXT_LENGTH, EMBED_DIM)

dataloader = create_dataloader(
    raw_text, batch_size=8, max_length=MAX_LENGTH,
    stride=MAX_LENGTH, shuffle=False
)

# Grab one batch and build input embeddings
for x_batch, _ in dataloader:
    token_embeds = token_embed_layer(x_batch)
    pos_embeds   = pos_embed_layer(torch.arange(MAX_LENGTH))
    input_embeds = token_embeds + pos_embeds  # (batch, seq_len, embed_dim)
    break

print("input_embeds shape:", input_embeds.shape)  # (8, 4, 256)

## Part 2A: MHA via Wrapper (stack of single heads)

The intuitive approach: run `num_heads` separate `CausalAttention` modules and concatenate their outputs.
Simple to understand, but less efficient than the weight-split approach below.

In [ ]:
class CausalAttention(nn.Module):
    """Single-head causal self-attention with dropout."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, qkv_bias: bool = False) -> None:
        super().__init__()
        self.d_out   = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Upper-triangular mask: position (i,j)=1 means token j is "future" for token i
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, _ = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values


class MultiHeadAttentionWrapper(nn.Module):
    """Stacks num_heads CausalAttention modules; output dim = d_out * num_heads."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        # Optional output projection to mix head outputs
        self.out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        concatenated = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(concatenated)


torch.manual_seed(123)
NUM_HEADS = 2
d_in      = EMBED_DIM         # 256
d_out_per_head = d_in // NUM_HEADS  # 128 per head → 256 total output

mha_wrapper = MultiHeadAttentionWrapper(
    d_in, d_out_per_head, context_length=MAX_LENGTH, dropout=0.0, num_heads=NUM_HEADS
)
out_wrapper = mha_wrapper(input_embeds)
print("Wrapper output shape:", out_wrapper.shape)  # (8, 4, 256)

## Part 2B: Efficient MHA (weight splits)

The production-grade approach: one big `W_query/key/value` matrix, then `view` + `transpose` to split across heads. Single matrix multiply covers all heads simultaneously — more parallelism-friendly on GPU.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Efficient MHA using tensor reshaping to split heads. Output dim = d_out."""

    def __init__(self, d_in: int, d_out: int, context_length: int,
                 dropout: float, num_heads: int, qkv_bias: bool = False) -> None:
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out     = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads  # dimension seen by each head

        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # standard output projection
        self.dropout  = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n_tokens, d_in = x.shape

        # Project then reshape: (b, n_tokens, d_out) → (b, n_heads, n_tokens, head_dim)
        def split_heads(proj: torch.Tensor) -> torch.Tensor:
            return proj.view(b, n_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        queries = split_heads(self.W_query(x))
        keys    = split_heads(self.W_key(x))
        values  = split_heads(self.W_value(x))

        # Scaled dot-product attention for all heads simultaneously
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Merge heads back: (b, n_heads, n_tokens, head_dim) → (b, n_tokens, d_out)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, n_tokens, self.d_out)
        return self.out_proj(context_vec)


torch.manual_seed(123)
mha_efficient = MultiHeadAttention(
    d_in=EMBED_DIM, d_out=EMBED_DIM,
    context_length=MAX_LENGTH, dropout=0.0, num_heads=NUM_HEADS
)
out_efficient = mha_efficient(input_embeds)
print("Efficient MHA output shape:", out_efficient.shape)  # (8, 4, 256)

## My Comparison

| | Wrapper approach | Weight-split approach |
|---|---|---|
| **Readability** | ✅ Easier to follow | Harder to follow |
| **Efficiency** | Slower (sequential heads) | ✅ Faster (parallel via batched matmul) |
| **Output dim** | `d_out * num_heads` | `d_out` (controlled directly) |
| **Used in practice** | Rarely | ✅ Standard in real LLMs |

**My takeaway**: I understood the wrapper approach first, then the weight-split fell into place. The `view`+`transpose` trick is the key — it reshapes without copying data so all heads compute attention in one batched operation.